# User Guide for performing calculations with physical quantities

This guide provides a step-by-step walkthrough of the main features for performing calculations with `paptools`. The core of `paptools` is the `Number` class. It represents a physical quantity with a value, an uncertainty (error), a unit, and a symbolic representation.

## 1. Working with Numbers
To create a measured value, simply import `Number` and provide the value and its uncertainty.

In [577]:
from paptools import Number

# A length of 5.0 meters with an uncertainty of 0.1 meters, the unit is "m" (meters), and the symbol used in equations is "L"
L = Number(5.0, 0.1, unit="m", symbol="L")

print(L)

5.0 ± 0.1  m


To see the symbolic representation of a `Number`, you can write the variable name at the end of a Jupyter notebook cell.

In [578]:
L

You can also print out the latex representation of the `Number`'s value and uncertainty using the `get_expr()` and `get_err_expr()` methods.

In [579]:
print(L.get_expr())
print(L.get_err_expr())

L
\Delta{L}


After creating `Number` instances, you can perform arithmetic operations such as addition, subtraction, multiplication, and division. The uncertainties will be propagated automatically according to Gaussian error propagation rules. If you do not provide an undertainty for a `Number`, it will be treated as having zero uncertainty and not be considered in the error propagation.

In [580]:
w = Number(3.0, 0.2, unit="m", symbol="W")

# Creating a variable x with an uncertainty. Simple LaTeX formatting is supported in the symbol.
x_i = Number(5.0, 0.1, unit="m", symbol=r"x_{\text{i}}")

# Creating a variable y_0 without uncertainty
y_0 = Number(3.0, unit="m", symbol=r"y_0")

# Performing arithmetic operations
z = (x_i + y_0) * w

# Displaying the well formatted equation and its error formula
z

In [581]:
# Displaying the LaTeX representation of the value and uncertainty
print(z.get_expr())
print(z.get_err_expr())

print("----------------------------------")
# Displaying the numerical result with uncertainty
print(z)

W \left(x_{\text{i}} + y_{0}\right)
\sqrt{W^{2} \Delta{x_{\text{i}}}^{2} + \Delta{W}^{2} \left(x_{\text{i}} + y_{0}\right)^{2}}
----------------------------------
24.0 ± 1.6278820596099708  m**2


As shown above, you can easily retrieve both the LaTeX representation and the numerical result with uncertainty of the computed `Number`. Since the perfomed operation consisted of a multiplication of two lengths, the resulting unit is in square meters (m²). By using the `convert_unit_to()` method, you can convert the result to a different compatible unit, such as square centimeters (cm²).

In [582]:
z = z.convert_unit_to("cm**2")
print(z)

240000.0 ± 16278.820596099707  cm**2


A simple `round()` method is provided to round the value and uncertainty according to the following convention: The uncertainty is rounded to one significant figure, if the first significant digit is 3 or greater, it is rounded to one significant figure if the first significant digit is 1 or 2. Uncertainties are always rounded up. The value is rounded to the same decimal place as the uncertainty. Values are rounded down if the next digit is 4 or less, otherwise rounded up.

In [583]:
z = z.round() # The round method does not modify the Number in place, it returns a new Number
print(z)

240000.0 ± 17000  cm**2


Trying to perform operations with incompatible units will raise an error.

In [584]:
x = Number(10.0, 0.5, unit="s", symbol="x")  # 10.0 seconds with 0.5 seconds uncertainty
y = Number(2.0, 0.1, unit="kg", symbol="y")   # 2.0 kilograms with 0.1 kilograms uncertainty

try:
    result = x + y  # This should raise an error due to incompatible units
except ValueError as e:
    print(f"Error: {e}")

Error: The <ufunc 'add'> operator for unyt_arrays with units 's' (dimensions '(time)') and 'kg' (dimensions '(mass)') is not well defined.


U can also convert units composed of multiple base units to other compatible units.

In [585]:
power = Number(0.2, 0.01, symbol="P", unit="nW")  # 0.2 Nano Watts with 0.01 Nano Watts uncertainty
time = Number(5.34, 0.1, symbol="t", unit="ns")  # 5.34 nanoseconds with 0.1 nanoseconds uncertainty

energy = power * time  # Energy in nanowatt-nanoseconds
print(energy)

energy_joules = energy.convert_unit_to("J")  # Convert to Joules
print(energy_joules)

energy_joules_rounded = energy_joules.round()  # Round the result
print(energy_joules_rounded)

energy_electronvolts = energy_joules.convert_unit_to("eV") # Convert to Electronvolts
print(energy_electronvolts)

energy_electronvolts_rounded = energy_electronvolts.round()  # Round the result
print(energy_electronvolts_rounded)
energy_electronvolts

1.068 ± 0.057022451718599404  nW*ns
1.0680000000000002e-18 ± 5.70224517185994e-20  J
1.07e-18 ± 5.999999999999999e-20  J
6.665931991083516 ± 0.35590616584365814  eV
6.7 ± 0.4  eV


If you keep working with previous results of calculations, the next operation will use the symbolic representation of the previous result. This way, you can build complex expressions step by step while keeping track of the uncertainties and units.

In [586]:
energy_2 = Number(2500, 0.1, symbol="E_2", unit="meV")  # 2.5 milli Electronvolts with 0.1 milli Electronvolts uncertainty

energy_3 = energy_electronvolts + energy_2

print(energy_3)
energy_3

9165.931991083517 ± 355.9061798923047  meV


Instead, you can also create a new variable from the result with a new symbol using the `set_symbol()` method.

In [587]:
energy_1 = energy_electronvolts.set_symbol("E_1") # Create a new variable with a new symbol from the previous result. The set_symbol method does not modify the Number in place, it returns a new Number.
energy_2 = Number(2600, 123, symbol="E_2", unit="meV")  # 2600 milli Electronvolts with 123 milli Electronvolts uncertainty

energy_3 = (energy_1 + energy_2).convert_unit_to("J").round() # Convert to Joules and round the result

print(energy_3)
energy_3


1.48e-18 ± 7e-20  J


## 2. Working with Arrays

The other main class in the `paptools` library is the `Array` class. It represents an array of physical quantities with values, uncertainties, units, and symbolic representations. The `Array` class inherits from the `Number` class, so it has all the same methods and properties as `Number`, but it also provides additional methods for working with arrays of physical quantities. To create an `Array`, import it and simply provide a list of values and a list of uncertainties, along with the unit and symbol.

In [588]:
from paptools import Array 

x_val = [1.0, 2.0, 3.0]  # Values for the array
x_err = [0.1, 0.2, 0.3]  # Uncertainties for the array

array_x = Array(x_val, x_err, unit="m", symbol="x") # An array of three values with their uncertainties, the unit is "m" (meters), and the symbol used in equations is "x"
print(array_x)

[1. 2. 3.] ± [0.1 0.2 0.3]  m


Instead of passing lists of values and uncertainties, you can also pass numpy arrays.

In [589]:
import numpy as np

y_val = np.array([4.0, 5.0, 6.0])  # Values for the array
y_err = np.array([0.2, 0.3, 0.4])  # Uncertainties for the array

array_y = Array(y_val, y_err, unit="m", symbol="y") # Create an Array using numpy arrays for values and uncertainties
print(array_y)

[4. 5. 6.] ± [0.2 0.3 0.4]  m


If all values in the array have the same uncertainty, you can also pass a single value for the uncertainty instead of a list or array.

In [590]:
z_val = np.array([1.0, 2.0, 3.0])  # Values for the array
z_err = 0.1  # Uncertainty for all values in the array

array_z = Array(z_val, z_err, unit="m", symbol="z") # Create an Array with a single uncertainty value for all entries
print(array_z)

[1. 2. 3.] ± [0.1 0.1 0.1]  m


Calculations with `Array` instances work similarly to calculations with `Number` instances, but they are performed element-wise on the arrays. The uncertainties will be propagated automatically according to Gaussian error propagation rules, just like with `Number` instances. You can mix `Array` instances with `Number` instances in calculations, and the operations will be performed element-wise, with the `Number` being treated as a constant value across the array.

In [591]:
a = Array([1.0, 2.0, 3.0], [0.1, 0.2, 0.3], unit="m", symbol="a")
b = Number(2.0, 0.1, unit="m", symbol="b")

c = a + b  
print(c)

c_rounded = c.round()
print(c_rounded)

c


[3. 4. 5.] ± [0.14142136 0.2236068  0.31622777]  m
[3. 4. 5.] ± [0.15 0.23 0.4 ]  m


You can access individual elements of an `Array` using indexing. The result will be a `Number` instance.

In [592]:
c_first_element = c[0]  # Accessing the first element of the array, which will be a Number instance
print(c_first_element)

3.0 ± 0.14142135623730953  m


You can also iterate over the elements of an `Array` using a for loop, which will yield `Number` instances for each element.

In [593]:
for element in c:
    print(f"Not rounded element: {element}, Rounded element: {element.round()}")

Not rounded element: 3.0 ± 0.14142135623730953  m, Rounded element: 3.0 ± 0.15  m
Not rounded element: 4.0 ± 0.223606797749979  m, Rounded element: 4.0 ± 0.23  m
Not rounded element: 5.0 ± 0.31622776601683794  m, Rounded element: 5.0 ± 0.4  m


You can get the length of an `Array` using the `len()` function, which will return the number of elements in the array.

In [594]:
print(f"Length of the array c: {len(c)}")

Length of the array c: 3
